In [1]:
import model_inference as mi
from importlib import reload
import pandas as pd
from data_extraction import main as extract_data
from feature_derivation import main as derive_features

# data_folder = extract_data()

# X_train, y_train, X_inference = derive_features(
#     data_folder=data_folder
# )

# X_train.to_parquet(f'{data_folder}/preprocessed_data/X_train.parquet')
# y_train.to_parquet(f'{data_folder}/preprocessed_data/y_train.parquet').to_frame(name='pct_change')
# X_inference.to_parquet(f'{data_folder}/preprocessed_data/X_inference.parquet')

data_folder = "Data_01_04_2026"

X_train = pd.read_parquet(f'{data_folder}/preprocessed_data/X_train.parquet')
y_train = pd.read_parquet(f'{data_folder}/preprocessed_data/y_train.parquet').close_pct_change
X_inference = pd.read_parquet(f'{data_folder}/preprocessed_data/X_inference.parquet')

In [56]:
params = {'num_parallel_tree': 10, "colsample_bytree": 0.8, "colsample_bynode": 0.8}

preds, metrics = mi.back_test(
    X_train,
    y_train,
    whole_back_test=True,
    **params
)

100%|██████████| 15/15 [1:34:27<00:00, 377.86s/it]


In [73]:
preds[
    (preds.close >= 0.25*preds.close_max)
    & (preds.marketcap >= preds.marketcap_quantile)
].groupby('calendardate').tail(20).groupby('calendardate').y_true.mean()

calendardate
2010-03-31    0.680747
2010-06-30    0.501756
2010-09-30   -0.039654
2010-12-31   -0.115710
2011-03-31    0.175443
2011-06-30    0.131082
2011-09-30    0.361904
2011-12-31    0.375998
2012-03-31    0.240352
2012-06-30    0.631356
2012-09-30    1.006107
2012-12-31    1.167736
2013-03-31    1.619219
2013-06-30    0.960039
2013-09-30    0.714242
2013-12-31    0.527144
2014-03-31    0.570265
2014-06-30    0.485469
2014-09-30    0.476616
2014-12-31    0.286693
2015-03-31    0.053341
2015-06-30   -0.193853
2015-09-30    0.292668
2015-12-31    0.454943
2016-03-31    0.854840
2016-06-30    0.804837
2016-09-30    0.551895
2016-12-31    0.644351
2017-03-31    0.911489
2017-06-30    0.974859
2017-09-30    0.969549
2017-12-31    0.535501
2018-03-31    0.984742
2018-06-30    0.500323
2018-09-30    0.369477
2018-12-31    1.665561
2019-03-31    0.904594
2019-06-30    1.313698
2019-09-30    2.958811
2019-12-31    2.596721
2020-03-31    1.985339
2020-06-30    1.592974
2020-09-30    1.04898

In [3]:
inference_predictions = mi.fit_and_predict(
    X_train,
    y_train,
    X_inference
)

In [4]:
reload(mi)
inference_performance_to_date = mi.obtain_inference_performance_to_date(
    inference_predictions,
    marketcap_quantile=0.25,
)

In [6]:
inference_predictions.to_parquet(f'{data_folder}/results/inference_predictions.parquet')

In [26]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
    & (inference_performance_to_date.calendardate == '2026-03-31')
].sort_values('pct_change').groupby('calendardate').tail(40)

,ticker,calendardate,pct_change,pct_change_at_max_date,close,close_max,marketcap,marketcap_quantile
1887,ARX,2026-03-31,0.597443,0.0,13.36,30.05,2.590386e+09,176145803.0
13490,TTAN,2026-03-31,0.619048,0.0,63.46,129.37,6.009078e+09,176145803.0
14645,YSS,2026-03-31,0.635288,0.0,22.17,33.95,2.688726e+09,176145803.0
7849,LIFE,2026-03-31,0.673350,0.0,11.17,16.85,7.484632e+08,176145803.0
1403,AMLX,2026-03-31,0.676981,0.0,13.90,40.93,1.559676e+09,176145803.0
3999,EIKN,2026-03-31,0.685377,0.0,10.58,16.26,5.240612e+08,176145803.0
2752,BLLN,2026-03-31,0.688935,0.0,78.94,130.18,3.096257e+09,176145803.0
14161,WLTH,2026-03-31,0.706330,0.0,9.25,14.19,1.319558e+09,176145803.0
5550,GLXY,2026-03-31,0.709559,0.0,18.45,42.86,8.536775e+09,176145803.0
6372,HTFL,2026-03-31,0.720080,0.0,24.33,39.91,2.184250e+09,176145803.0


In [32]:
selected_stocks = [
    "NTSK",
    "INV",
    "MNTN",
    "CHA",
    "AGBK",
    "KLAR",
    "SSII",
    "BETA",
    "WOLF",
    "FLY",
    "EQPT",
    "WYFI",
    "OMDA",
    "TLX",
    "IBTA",
    "FIGR",
    "BBNX",
    "MANE",
    "LMRI",
    "PICS",
    "SAIL",
    "ANTA",
    "XZO",
    "ETOR",
    "PTRN",
    "AERO",
    "HTFL",
    "GLXY",
    "WLTH",
    "BLLN"
]

len(selected_stocks)

30

In [22]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
].sort_values('pct_change').groupby('calendardate').tail(20).groupby('calendardate').pct_change_at_max_date.mean()

calendardate
2025-06-30    2.654480
2025-09-30    0.551877
2025-12-31    0.049873
2026-03-31    0.000000
Name: pct_change_at_max_date, dtype: float64

In [5]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
].sort_values('pct_change').groupby('calendardate').tail(20).groupby('calendardate').pct_change_at_max_date.mean()

calendardate
2025-03-31    2.134912
2025-06-30    2.092430
2025-09-30    0.568499
2025-12-31    0.217706
Name: pct_change_at_max_date, dtype: float64